# Scope IAM roles and S3 bucket policies for a multi-team data lake

The compliance team filed an audit finding this morning. The analytics IAM role on your shared data lake bucket can read every team's prefix, including the ML team's training datasets and compliance's audit-log exports. Three teams sharing one bucket, one role with universal read, no isolation. The CTO wants this scoped before end of day and proven via a deny-test before any new credentials are handed out.

You have one deliverable: build a least-privilege IAM role per team (analytics, ML, compliance), scope each role's inline policy to its own S3 prefix, layer a defense-in-depth bucket policy that explicitly denies cross-team access even if an inline policy is too permissive, and run a programmatic deny-test that proves the analytics role cannot read the ML prefix. When you finish, every team has credentials they can actually trust, and you have a cleanup script that removes every resource you created.

The three teams use the same bucket but never the same prefix:
- **analytics/** payment-funnel reports the analytics team owns.
- **ml/** training datasets the ML team owns.
- **compliance/** audit-log exports the compliance team owns.

Each team is supposed to be locked to its own prefix. The deny-test is where you prove it.

**Time.** Plan for about 50 minutes hands-on. If you hit IAM propagation snags or trust-policy debugging, budget up to 90 minutes. The cleanup cell at the bottom keeps your AWS costs near zero either way.

**Cost.** This lab costs nothing if you follow the steps and clean up. IAM operations are free at any volume. STS calls are free. The S3 footprint of 1 bucket and 3 tiny seed objects is well under the always-free tier. The cleanup cell at the end tears down every resource so there is nothing to forget about.

This lab maps to AWS DEA-C01 Domain 4: Data Security and Governance (18% of exam weight).

In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT.md
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.3.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT.md
# section 12.

import atexit
import getpass
import json
import time

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)

LAB_ID = "iam-data-lake-permissions"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[1].slug exactly
REGION = "us-east-1"  # all DEA-C01 labs run in us-east-1 per LAB-CREATION-BLUEPRINT section 15

# Per-team identifiers. Each team's role name and S3 prefix share the same
# short name. Adding a fourth team later means appending to TEAMS only.
TEAMS = ["analytics", "ml", "compliance"]
ROLE_NAMES = {team: f"labrun-{LAB_ID}-{team}-role" for team in TEAMS}
PREFIXES = {team: f"{team}/" for team in TEAMS}

# Bucket name is resolved in the next cell once we know the AWS account ID.
# S3 bucket names must be globally unique, so we suffix with the account ID.
BUCKET_NAME = None

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# and confirm the region. This cell must succeed before the manifest cell
# creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"All DEA-C01 labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

# Validate credentials against AWS via STS GetCallerIdentity. Fail fast with a
# clear message rather than waiting for the first IAM call to error out.
sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")
print(f"Session token in use: {bool(aws_session_token)}")

# Resolve the bucket name now that we know the account ID. S3 bucket names
# must be globally unique.
BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
print(f"Bucket name resolved: {BUCKET_NAME}")

# Register the session with labrun-checks. register() returns None;
# do not assign its return value.
register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")


In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# The manifest is module-level and in reverse-creation order. Lab 02 has no
# critical (hourly-billed) resources, so reverse-creation order is sufficient.
# Per RESOURCE-SAFETY-SPEC Rule 4, the orphan scan blocks execution if any
# tagged resources from a prior session exist (not just print a warning).

CLEANUP_MANIFEST = [
    CleanupResource(
        type="iam_role",
        id=ROLE_NAMES["compliance"],
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {ROLE_NAMES['compliance']}",
    ),
    CleanupResource(
        type="iam_role",
        id=ROLE_NAMES["ml"],
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {ROLE_NAMES['ml']}",
    ),
    CleanupResource(
        type="iam_role",
        id=ROLE_NAMES["analytics"],
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {ROLE_NAMES['analytics']}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    """Best-effort teardown on kernel shutdown.

    Runs run_cleanup against the manifest. Errors are swallowed because
    atexit handlers must not raise; the dedicated cleanup cell prints the
    full structured report and is the authoritative cleanup entry point.
    """
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    """Refuse to start if a previous run left tagged resources behind.

    Per RESOURCE-SAFETY-SPEC Rule 4, the setup cell must check for leftover
    state with the lab's tag and surface the cleanup command before creating
    any new resources. This prevents duplicate-resource scenarios.
    """
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate resources.")
    print("Run the cleanup cell at the bottom of this notebook first, or")
    print("remove them manually with the AWS CLI commands printed by the")
    print("cleanup cell, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")


## Task 1: Create the data lake bucket and seed each team prefix

S3 has no real folders. The three "team prefixes" exist only as side effects of object keys that start with `analytics/`, `ml/`, or `compliance/`. To make the bucket usable for the deny-test later, you need at least one object under each prefix.

Build it in this order:
1. Create the bucket (`labrun-iam-data-lake-permissions-{your-account-id}`) and tag it with `labrun:lab-slug=iam-data-lake-permissions` at creation. The cleanup cell's tag audit needs that tag to find the bucket.
2. Upload three tiny seed objects, one per team. Use any one-line content; the bytes do not matter, only the keys do.

The lab is not about the data. It is about who can read what. The seed objects exist so that `s3:ListObjectsV2` in the deny-test has something non-empty to either find or be denied.


In [ ]:
# NBVAL_SKIP
# Task 1: Create the bucket, seed each team prefix, tag the bucket with
# labrun:lab-slug.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Create the bucket.
# YOUR CODE: Create the S3 bucket using s3.create_bucket(Bucket=BUCKET_NAME).
# This lab runs in us-east-1, where create_bucket does NOT accept
# CreateBucketConfiguration. Outside us-east-1 you would also pass
# CreateBucketConfiguration={"LocationConstraint": REGION}.

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket created and tagged: {BUCKET_NAME}")

# Seed one tiny placeholder object per team prefix. The content is arbitrary;
# only the prefix in the Key matters.
for team in TEAMS:
    key = f"{PREFIXES[team]}seed.txt"
    body = f"placeholder seed for {team} team\n".encode("utf-8")
    # YOUR CODE: Upload the placeholder object using s3.put_object() with
    # Bucket=BUCKET_NAME, Key=key, and Body=body.
    print(f"Uploaded s3://{BUCKET_NAME}/{key}")

print()
print(f"All three team prefixes seeded under s3://{BUCKET_NAME}/")


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Bucket exists with the lab tag, and each team prefix has at
# least one object.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        # Bucket exists.
        try:
            s3_client.head_bucket(Bucket=BUCKET_NAME)
        except ClientError as e:
            error_code = e.response["Error"]["Code"]
            if error_code in ("404", "NoSuchBucket"):
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Bucket {BUCKET_NAME} does not exist.",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        # Bucket has the lab tag.
        try:
            tag_resp = s3_client.get_bucket_tagging(Bucket=BUCKET_NAME)
            tags = {t["Key"]: t["Value"] for t in tag_resp.get("TagSet", [])}
        except ClientError as e:
            if e.response["Error"]["Code"] == "NoSuchTagSet":
                tags = {}
            else:
                return CheckpointResult(status="error", error_reason=str(e))
        if tags.get(LAB_TAG_KEY) != LAB_TAG_VALUE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Bucket {BUCKET_NAME} is missing tag "
                    f"{LAB_TAG_KEY}={LAB_TAG_VALUE}. Found tags: {tags}"
                ),
            )

        # Each team prefix has at least one object.
        empty_prefixes = []
        for team in TEAMS:
            resp = s3_client.list_objects_v2(
                Bucket=BUCKET_NAME, Prefix=PREFIXES[team], MaxKeys=1
            )
            if not resp.get("Contents"):
                empty_prefixes.append(PREFIXES[team])
        if empty_prefixes:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"These prefixes have no objects: {', '.join(empty_prefixes)}. "
                    f"Each team prefix needs at least one seed object."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint says one or more team prefixes is empty. S3 has no real folders. Creating a "directory" is just uploading an object whose key starts with that prefix. Most likely the bucket exists but you skipped the seed-object upload for at least one team.

</details>

<details><summary>Hint 2 (direction)</summary>

S3 prefixes only exist as side effects of object keys. You need three separate `s3.put_object` calls, one per team, with keys like `analytics/seed.txt`, `ml/seed.txt`, `compliance/seed.txt`. The seed contents can be anything; the prefix in the Key is what matters.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 1:

```python
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Create the bucket. us-east-1 rejects LocationConstraint; other regions require it.
if REGION == "us-east-1":
    s3.create_bucket(Bucket=BUCKET_NAME)
else:
    s3.create_bucket(
        Bucket=BUCKET_NAME,
        CreateBucketConfiguration={"LocationConstraint": REGION},
    )

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket created and tagged: {BUCKET_NAME}")

for team in TEAMS:
    key = f"{PREFIXES[team]}seed.txt"
    body = f"placeholder seed for {team} team\n".encode("utf-8")
    s3.put_object(Bucket=BUCKET_NAME, Key=key, Body=body)
    print(f"Uploaded s3://{BUCKET_NAME}/{key}")

print()
print(f"All three team prefixes seeded under s3://{BUCKET_NAME}/")
```

If the bucket already exists from a prior run that did not clean up, the orphan scan in setup will tell you to run cleanup first.

</details>

**Wallet check.** Still at $0.00. Bucket creation, three tiny seed objects, and tagging fit inside the always-free S3 tier. Your coffee this morning cost infinitely more.


## Task 2: Create three IAM roles, each scoped to a single team prefix

The compliance audit fails because one role has access to everything. The fix is one role per team, with two pieces correctly aligned: a trust policy that controls who can assume the role, and an inline policy that controls what the role can do once assumed.

Build it once per team:
1. Create the IAM role with a trust policy that allows the current AWS account to call `sts:AssumeRole` on it. Use the account-root principal `arn:aws:iam::{account-id}:root` so any identity in your account with `sts:AssumeRole` permission can assume the role. This is what the deny-test in Task 4 needs.
2. Attach an inline policy that grants `s3:GetObject` and `s3:PutObject` on the team's prefix path (e.g., `arn:aws:s3:::{bucket}/analytics/*`), plus `s3:ListBucket` on the bucket itself with a `Condition` block that pins `s3:prefix` to the team's prefix.
3. Tag the role with `labrun:lab-slug=iam-data-lake-permissions` so the cleanup audit can find it.

Three roles. Three trust policies. Three inline policies. After this task, each role can only see and write to its own team prefix. The cell ends with a 15-second IAM propagation wait because `sts.assume_role` is stricter than most other IAM calls.


In [ ]:
# NBVAL_SKIP
# Task 2: Create three IAM roles, attach an inline policy to each scoped to
# that team's prefix only.

iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,  # IAM is global; boto3 still wants a region for endpoint resolution
)

# Trust policy: any identity in this account with sts:AssumeRole permission
# can assume any of these roles. The deny-test in Task 4 will assume each role.
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:root"},
            "Action": "sts:AssumeRole",
        }
    ],
}

for team in TEAMS:
    role_name = ROLE_NAMES[team]
    prefix = PREFIXES[team]

    # Create the role.
    try:
        # YOUR CODE: Create the IAM role using iam.create_role() with RoleName,
        # AssumeRolePolicyDocument, Description, and Tags. Use role_name and
        # json.dumps(trust_policy). Tags format:
        # [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
        print(f"Created role: {role_name}")
    except ClientError as e:
        if e.response["Error"]["Code"] == "EntityAlreadyExists":
            print(f"Role {role_name} already exists, reusing.")
        else:
            raise

    # Inline policy: scoped to this team's prefix only. GetObject and PutObject
    # are scoped via the Resource ARN. ListBucket is scoped via a s3:prefix
    # Condition because the ListBucket Resource is the bucket itself, not the
    # prefix.
    inline_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": ["s3:GetObject", "s3:PutObject"],
                "Resource": f"arn:aws:s3:::{BUCKET_NAME}/{prefix}*",
            },
            {
                "Effect": "Allow",
                "Action": "s3:ListBucket",
                "Resource": f"arn:aws:s3:::{BUCKET_NAME}",
                "Condition": {
                    "StringLike": {"s3:prefix": [f"{prefix}*"]}
                },
            },
        ],
    }
    # YOUR CODE: Attach inline_policy to the role using iam.put_role_policy()
    # with RoleName=role_name, PolicyName=f"labrun-{LAB_ID}-{team}-prefix-access",
    # and PolicyDocument=json.dumps(inline_policy).
    print(f"Attached inline policy scoped to s3://{BUCKET_NAME}/{prefix}*")

# IAM propagation. AWS docs note "may take a few seconds" before roles and
# policies are usable. The deny-test in Task 4 calls sts.assume_role, which is
# stricter than most other IAM-dependent calls. 15 seconds covers nearly all
# observed propagation cases in us-east-1.
print()
print("Three roles created and policies attached. Letting AWS propagate...")
time.sleep(15)
print("IAM propagation buffer elapsed. Roles should be assumable now.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: All three roles exist with the account-root trust policy and an
# inline policy scoped to the correct team prefix.

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        iam_client = boto3.client(
            "iam",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        expected_principal = f"arn:aws:iam::{ACCOUNT_ID}:root"
        required_actions = {"s3:GetObject", "s3:PutObject", "s3:ListBucket"}
        action_wildcards = {"s3:*", "*"}

        for team in TEAMS:
            role_name = ROLE_NAMES[team]
            prefix = PREFIXES[team]

            # Role exists.
            try:
                role_resp = iam_client.get_role(RoleName=role_name)
            except ClientError as e:
                if e.response["Error"]["Code"] == "NoSuchEntity":
                    return CheckpointResult(
                        status="fail",
                        error_reason=f"Role {role_name} does not exist.",
                    )
                return CheckpointResult(status="error", error_reason=str(e))

            # Trust policy contains the account-root principal.
            trust = role_resp["Role"]["AssumeRolePolicyDocument"]
            trust_principals = []
            for stmt in trust.get("Statement", []):
                p = stmt.get("Principal", {})
                aws = p.get("AWS")
                if isinstance(aws, str):
                    trust_principals.append(aws)
                elif isinstance(aws, list):
                    trust_principals.extend(aws)
            if expected_principal not in trust_principals:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Role {role_name} trust policy is missing AWS principal "
                        f"{expected_principal!r}. Found: {trust_principals}"
                    ),
                )

            # Exactly one inline policy attached.
            inline_resp = iam_client.list_role_policies(RoleName=role_name)
            policy_names = inline_resp.get("PolicyNames", [])
            if not policy_names:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Role {role_name} has no inline policy attached.",
                )

            # Inline policy references the correct prefix Resource and s3:prefix
            # Condition value. Substring match is sufficient for the lab; the
            # student's policy structure can vary as long as both strings appear.
            doc_resp = iam_client.get_role_policy(
                RoleName=role_name, PolicyName=policy_names[0]
            )
            policy_doc = doc_resp["PolicyDocument"]
            policy_str = json.dumps(policy_doc)
            expected_resource = f"arn:aws:s3:::{BUCKET_NAME}/{prefix}*"
            expected_prefix_value = f"{prefix}*"
            if expected_resource not in policy_str:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Role {role_name} inline policy does not include the "
                        f"prefix-scoped Resource ARN {expected_resource!r}. "
                        f"Check the s3:GetObject/s3:PutObject Resource."
                    ),
                )
            if expected_prefix_value not in policy_str:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Role {role_name} inline policy is missing an s3:prefix "
                        f"Condition value of {expected_prefix_value!r}. "
                        f"ListBucket must be scoped via Condition.StringLike on s3:prefix."
                    ),
                )

            # Required S3 actions are explicitly listed (or covered by s3:* / *).
            all_actions = set()
            for stmt in policy_doc.get("Statement", []):
                actions = stmt.get("Action", [])
                if isinstance(actions, str):
                    all_actions.add(actions)
                else:
                    all_actions.update(actions)
            has_wildcard = bool(all_actions & action_wildcards)
            missing = required_actions - all_actions
            if missing and not has_wildcard:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Role {role_name} inline policy is missing required "
                        f"actions: {sorted(missing)}. Add them explicitly or "
                        f"use the s3:* wildcard."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)


<details><summary>Hint 1 (nudge)</summary>

An IAM role does not work just by existing. Two pieces have to line up: the trust policy (who can assume this role) and the inline policy (what the role is allowed to do once assumed). The checkpoint is failing because at least one of those two is missing or wrong on at least one role.

</details>

<details><summary>Hint 2 (direction)</summary>

The trust policy `Principal` must be the account-root ARN: `arn:aws:iam::{ACCOUNT_ID}:root`. That allows any identity in your account with `sts:AssumeRole` permission to assume the role, which is what the deny-test in Task 4 needs. The inline policy needs `s3:GetObject`, `s3:PutObject`, and `s3:ListBucket` actions scoped to the team prefix. `s3:GetObject` and `s3:PutObject` are scoped via the `Resource` ARN (e.g., `arn:aws:s3:::{BUCKET_NAME}/analytics/*`); `s3:ListBucket` is scoped via a `Condition` block on `s3:prefix`, not the Resource.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 2:

```python
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"AWS": f"arn:aws:iam::{ACCOUNT_ID}:root"},
            "Action": "sts:AssumeRole",
        }
    ],
}

for team in TEAMS:
    role_name = ROLE_NAMES[team]
    prefix = PREFIXES[team]

    try:
        iam.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description=f"labrun {LAB_ID} role for {team} team",
            Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
        )
        print(f"Created role: {role_name}")
    except ClientError as e:
        if e.response["Error"]["Code"] == "EntityAlreadyExists":
            print(f"Role {role_name} already exists, reusing.")
        else:
            raise

    inline_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": ["s3:GetObject", "s3:PutObject"],
                "Resource": f"arn:aws:s3:::{BUCKET_NAME}/{prefix}*",
            },
            {
                "Effect": "Allow",
                "Action": "s3:ListBucket",
                "Resource": f"arn:aws:s3:::{BUCKET_NAME}",
                "Condition": {
                    "StringLike": {"s3:prefix": [f"{prefix}*"]}
                },
            },
        ],
    }
    iam.put_role_policy(
        RoleName=role_name,
        PolicyName=f"labrun-{LAB_ID}-{team}-prefix-access",
        PolicyDocument=json.dumps(inline_policy),
    )
    print(f"Attached inline policy scoped to s3://{BUCKET_NAME}/{prefix}*")

print()
print("Three roles created and policies attached. Letting AWS propagate...")
time.sleep(15)
print("IAM propagation buffer elapsed. Roles should be assumable now.")
```

</details>

**Wallet check.** Still at $0.00. IAM operations are free at any volume. Three roles, three inline policies, zero metered API calls. The 15-second propagation wait does not bill against anything.


## Task 3: Layer a defense-in-depth bucket policy with cross-team Deny statements

The inline policies you attached in Task 2 already restrict each role to its own prefix. But policy drift happens. A future engineer might widen the analytics role's inline policy to "fix" a quick request and accidentally re-grant cross-team access. A bucket policy with explicit Deny statements blocks the bad cases regardless of how loose any role's identity policy gets.

For each team role, write a Deny statement that:
- Names the role's full ARN (`arn:aws:iam::{ACCOUNT_ID}:role/{role-name}`) as the `Principal`.
- Denies `s3:GetObject` on the OTHER two teams' prefix object ARNs (`arn:aws:s3:::{bucket}/{other-prefix}*`).

Three Deny statements total. Each role appears as the `Principal` of exactly one Deny statement covering the two prefixes it must never read. Bucket policies apply at request time, so no IAM-style propagation wait is needed.


In [ ]:
# NBVAL_SKIP
# Task 3: Apply a bucket policy with three Deny statements, one per team role.
# Each role is explicitly denied s3:GetObject on the other teams' prefixes.
# Bucket policies apply at request time; no propagation wait needed.

bucket_policy_statements = []
for team in TEAMS:
    role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAMES[team]}"
    other_prefixes = [PREFIXES[other] for other in TEAMS if other != team]
    other_prefix_resources = [
        f"arn:aws:s3:::{BUCKET_NAME}/{p}*" for p in other_prefixes
    ]
    bucket_policy_statements.append({
        "Sid": f"Deny{team.capitalize()}CrossTeamRead",
        "Effect": "Deny",
        "Principal": {"AWS": role_arn},
        "Action": "s3:GetObject",
        "Resource": other_prefix_resources,
    })

bucket_policy = {
    "Version": "2012-10-17",
    "Statement": bucket_policy_statements,
}

# YOUR CODE: Apply the bucket_policy using s3.put_bucket_policy() with
# Bucket=BUCKET_NAME and Policy=json.dumps(bucket_policy).

print(f"Bucket policy applied to {BUCKET_NAME} with {len(bucket_policy_statements)} Deny statements:")
for stmt in bucket_policy_statements:
    print(f"  - {stmt['Sid']}")
    print(f"      Principal: {stmt['Principal']['AWS']}")
    print(f"      Resources: {stmt['Resource']}")


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: Bucket policy exists with three Deny statements that block
# each role from reading the other teams' prefixes.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            policy_resp = s3_client.get_bucket_policy(Bucket=BUCKET_NAME)
        except ClientError as e:
            if e.response["Error"]["Code"] == "NoSuchBucketPolicy":
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Bucket {BUCKET_NAME} has no bucket policy attached.",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        policy_doc = json.loads(policy_resp["Policy"])
        statements = policy_doc.get("Statement", [])

        # Per-team check: each role must be the Principal of a Deny statement
        # that covers the OTHER teams' prefix object ARNs.
        action_wildcards = {"s3:*", "*"}
        for team in TEAMS:
            role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAMES[team]}"
            other_prefixes = [PREFIXES[other] for other in TEAMS if other != team]
            other_resources = {
                f"arn:aws:s3:::{BUCKET_NAME}/{p}*" for p in other_prefixes
            }

            matching_deny = None
            for stmt in statements:
                if stmt.get("Effect") != "Deny":
                    continue
                p = stmt.get("Principal", {})
                aws_principal = p.get("AWS")
                if isinstance(aws_principal, str):
                    principals = [aws_principal]
                elif isinstance(aws_principal, list):
                    principals = aws_principal
                else:
                    principals = []
                if role_arn not in principals:
                    continue
                actions = stmt.get("Action", [])
                if isinstance(actions, str):
                    actions = [actions]
                # Accept explicit s3:GetObject OR a wildcard that covers it.
                action_set = set(actions)
                if not ("s3:GetObject" in action_set or action_set & action_wildcards):
                    continue
                resources = stmt.get("Resource", [])
                if isinstance(resources, str):
                    resources = [resources]
                if other_resources.issubset(set(resources)):
                    matching_deny = stmt
                    break

            if matching_deny is None:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Bucket policy is missing a Deny statement that names "
                        f"{role_arn!r} as Principal and denies s3:GetObject (or "
                        f"a wildcard covering it) on the other teams' prefixes. "
                        f"Expected Resources: {sorted(other_resources)}."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)


<details><summary>Hint 1 (nudge)</summary>

A bucket policy can deny actions even when a role's own inline policy allows them. AWS evaluates explicit denies before allows, so an explicit deny on cross-team prefix access closes the door regardless of what the role's identity policy says. The checkpoint fails when the bucket policy is missing one or more deny statements, or the Principal-to-Resource pairing is wrong.

</details>

<details><summary>Hint 2 (direction)</summary>

Each Deny statement needs the team's role ARN as the `Principal`, `s3:GetObject` as the `Action`, and the OTHER teams' prefix object ARNs as the `Resource`. You need three Deny statements, one per role, each blocking access to the other two teams' prefixes. The role ARN format is `arn:aws:iam::{ACCOUNT_ID}:role/{role-name}`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 3:

```python
bucket_policy_statements = []
for team in TEAMS:
    role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAMES[team]}"
    other_prefixes = [PREFIXES[other] for other in TEAMS if other != team]
    other_prefix_resources = [
        f"arn:aws:s3:::{BUCKET_NAME}/{p}*" for p in other_prefixes
    ]
    bucket_policy_statements.append({
        "Sid": f"Deny{team.capitalize()}CrossTeamRead",
        "Effect": "Deny",
        "Principal": {"AWS": role_arn},
        "Action": "s3:GetObject",
        "Resource": other_prefix_resources,
    })

bucket_policy = {
    "Version": "2012-10-17",
    "Statement": bucket_policy_statements,
}

s3.put_bucket_policy(
    Bucket=BUCKET_NAME,
    Policy=json.dumps(bucket_policy),
)
print(f"Bucket policy applied to {BUCKET_NAME} with {len(bucket_policy_statements)} Deny statements:")
for stmt in bucket_policy_statements:
    print(f"  - {stmt['Sid']}")
    print(f"      Principal: {stmt['Principal']['AWS']}")
    print(f"      Resources: {stmt['Resource']}")
```

</details>

**Wallet check.** Still at $0.00. Bucket policies are free. `PutBucketPolicy` is a free request. The meter has not budged since you started.


## Task 4: Run a deny-test that confirms cross-team isolation

The audit deliverable requires programmatic proof that the analytics role cannot read the ML team's prefix. A multiple-choice answer about IAM precedence does not satisfy compliance. They want an exception raised from S3, in this account, by a role assumed via STS.

The test pattern is:
1. Call `sts.assume_role` against the analytics role ARN with the lab's base credentials. Each test invocation must mint fresh temporary credentials. Reusing a cached AssumeRole response across permission updates is how false positives happen.
2. Build a NEW `boto3.client('s3', ...)` from the response's `Credentials` block. The original `s3` client uses your base credentials, which can read every prefix; the assumed-role client is the one that proves isolation.
3. Call `s3.get_object` on `ml/seed.txt`. Catch `ClientError`, inspect `e.response['Error']['Code']`, and treat anything other than `AccessDenied` as a test failure. AWS does not return an empty response for denies; it raises.
4. Call `s3.get_object` on `analytics/seed.txt`. Same role, this time on its own prefix. The call must succeed.

When both halves pass, the deny-test is the artifact you hand to compliance.


In [ ]:
# NBVAL_SKIP
# Task 4: Deny-test. Assume the analytics role with a fresh STS call,
# build a new boto3 s3 client from the temporary credentials, and prove
# cross-team access fails while same-team access succeeds.

analytics_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAMES['analytics']}"

# STS client uses the lab's base credentials, not any assumed credentials.
sts_client = boto3.client(
    "sts",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Fresh AssumeRole. No caching across runs.
# YOUR CODE: Call sts_client.assume_role() with RoleArn=analytics_role_arn and
# RoleSessionName="labrun-deny-test-analytics". Save the response to assume_resp.
temp_creds = assume_resp["Credentials"]
print(f"Assumed role: {analytics_role_arn}")
print(f"Temporary credentials expire at: {temp_creds['Expiration'].isoformat()}")

# YOUR CODE: Build a new boto3 s3 client from the assumed-role temporary
# credentials. Pass aws_access_key_id, aws_secret_access_key, and
# aws_session_token sourced from temp_creds (keys: AccessKeyId, SecretAccessKey,
# SessionToken) plus region_name=REGION. Save the client to analytics_s3.

# Half 1: cross-team read must be denied.
cross_team_passed = False
try:
    # YOUR CODE: Call analytics_s3.get_object() with Bucket=BUCKET_NAME and
    # Key="ml/seed.txt". The bucket policy should deny this and raise ClientError.
    print(f"FAIL: analytics role read s3://{BUCKET_NAME}/ml/seed.txt. Bucket policy is not blocking cross-team access.")
except ClientError as e:
    code_str = e.response["Error"]["Code"]
    if code_str == "AccessDenied":
        print(f"PASS: analytics role got AccessDenied on s3://{BUCKET_NAME}/ml/seed.txt as expected.")
        cross_team_passed = True
    else:
        print(f"FAIL: expected AccessDenied on the ml/ prefix, got {code_str} instead.")

# Half 2: same-team read must succeed.
same_team_passed = False
try:
    # YOUR CODE: Call analytics_s3.get_object() with Bucket=BUCKET_NAME and
    # Key="analytics/seed.txt". This call should succeed. Save the response to obj.
    body = obj["Body"].read().decode("utf-8")
    print(f"PASS: analytics role read s3://{BUCKET_NAME}/analytics/seed.txt. Body: {body!r}")
    same_team_passed = True
except ClientError as e:
    code_str = e.response["Error"]["Code"]
    print(f"FAIL: analytics role could not read its own prefix. Got {code_str}.")

print()
if cross_team_passed and same_team_passed:
    print("Deny-test result: cross-team isolation confirmed.")
else:
    print("Deny-test result: at least one half failed. Re-check the inline policy and bucket policy.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: Programmatic deny-test. Independently assume the analytics role,
# build a new s3 client from the temporary credentials, and verify GetObject is
# denied on the ml/ prefix and allowed on the analytics/ prefix.

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        sts_client = boto3.client(
            "sts",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAMES['analytics']}"
        try:
            assume_resp = sts_client.assume_role(
                RoleArn=role_arn,
                RoleSessionName="labrun-checkpoint-4",
            )
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Could not assume {role_arn}. STS returned "
                    f"{e.response['Error']['Code']}: {e.response['Error']['Message']}. "
                    f"Check the trust policy Principal."
                ),
            )

        temp_creds = assume_resp["Credentials"]
        s3_assumed = boto3.client(
            "s3",
            aws_access_key_id=temp_creds["AccessKeyId"],
            aws_secret_access_key=temp_creds["SecretAccessKey"],
            aws_session_token=temp_creds["SessionToken"],
            region_name=REGION,
        )

        # Cross-team read must raise AccessDenied.
        try:
            s3_assumed.get_object(Bucket=BUCKET_NAME, Key="ml/seed.txt")
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Analytics role successfully read s3://{BUCKET_NAME}/ml/seed.txt. "
                    f"The bucket policy is not denying cross-team GetObject."
                ),
            )
        except ClientError as e:
            code_str = e.response["Error"]["Code"]
            if code_str != "AccessDenied":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Expected AccessDenied on the ml/ prefix, got {code_str}. "
                        f"Verify the bucket policy Deny statement Principal and Resource."
                    ),
                )

        # Same-team read must succeed.
        try:
            s3_assumed.get_object(Bucket=BUCKET_NAME, Key="analytics/seed.txt")
        except ClientError as e:
            code_str = e.response["Error"]["Code"]
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Analytics role could not read its own prefix s3://{BUCKET_NAME}/analytics/seed.txt. "
                    f"Got {code_str}. Verify the inline policy Resource and that the bucket policy is not over-broad."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)


<details><summary>Hint 1 (nudge)</summary>

The deny-test relies on freshly assumed credentials, not your current session. If you reuse a boto3 client built from your own credentials, you are testing your own permissions, not the role's. Each test run needs to call `sts.assume_role` and build a new boto3 client from the response.

</details>

<details><summary>Hint 2 (direction)</summary>

Call `sts.assume_role` with the analytics role ARN to get temporary credentials. Build a new `boto3.client('s3', aws_access_key_id=..., aws_secret_access_key=..., aws_session_token=...)` from the response's `Credentials` block. Then call `s3.get_object` against the `ml/seed.txt` key. AWS raises `botocore.exceptions.ClientError` with `e.response['Error']['Code'] == 'AccessDenied'` for denied keys; it does NOT return an empty response. Wrap the call in `try / except ClientError`, inspect the error code, and treat anything other than `AccessDenied` as a test failure.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 4:

```python
analytics_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAMES['analytics']}"

sts_client = boto3.client(
    "sts",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

assume_resp = sts_client.assume_role(
    RoleArn=analytics_role_arn,
    RoleSessionName="labrun-deny-test-analytics",
)
temp_creds = assume_resp["Credentials"]
print(f"Assumed role: {analytics_role_arn}")
print(f"Temporary credentials expire at: {temp_creds['Expiration'].isoformat()}")

analytics_s3 = boto3.client(
    "s3",
    aws_access_key_id=temp_creds["AccessKeyId"],
    aws_secret_access_key=temp_creds["SecretAccessKey"],
    aws_session_token=temp_creds["SessionToken"],
    region_name=REGION,
)

cross_team_passed = False
try:
    analytics_s3.get_object(Bucket=BUCKET_NAME, Key="ml/seed.txt")
    print(f"FAIL: analytics role read s3://{BUCKET_NAME}/ml/seed.txt. Bucket policy is not blocking cross-team access.")
except ClientError as e:
    code_str = e.response["Error"]["Code"]
    if code_str == "AccessDenied":
        print(f"PASS: analytics role got AccessDenied on s3://{BUCKET_NAME}/ml/seed.txt as expected.")
        cross_team_passed = True
    else:
        print(f"FAIL: expected AccessDenied on the ml/ prefix, got {code_str} instead.")

same_team_passed = False
try:
    obj = analytics_s3.get_object(Bucket=BUCKET_NAME, Key="analytics/seed.txt")
    body = obj["Body"].read().decode("utf-8")
    print(f"PASS: analytics role read s3://{BUCKET_NAME}/analytics/seed.txt. Body: {body!r}")
    same_team_passed = True
except ClientError as e:
    code_str = e.response["Error"]["Code"]
    print(f"FAIL: analytics role could not read its own prefix. Got {code_str}.")

print()
if cross_team_passed and same_team_passed:
    print("Deny-test result: cross-team isolation confirmed.")
else:
    print("Deny-test result: at least one half failed. Re-check the inline policy and bucket policy.")
```

</details>

**Wallet check.** Still at $0.00. STS AssumeRole calls are free for IAM-based identity. Two GetObject requests against tiny placeholder objects fit comfortably inside the always-free S3 request quota. The deny-test does not move the meter.


In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order. Lab 02 has no critical (hourly-billed) resources, so the canonical
# summary always reports zero critical destructions.

import sys

result = run_cleanup(CLEANUP_MANIFEST)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)


**Session total: $0.00.** IAM, STS, bucket policies, and a handful of S3 requests against tiny placeholder objects all fit under the always-free tier. The bucket and roles were destroyed by the cleanup cell above, so nothing is still accruing. Your coffee this morning cost infinitely more than the AWS bill from this session.


## These are not graded. They are for you.

Three questions worth sitting with before the exam.

1. If you removed the bucket policy and left the three inline policies in place, what would change for the analytics role? Identify which permission boundary is doing the work in each layer of the design, and what the failure mode looks like when one layer drifts. This maps to DEA-C01 Domain 4 expectations on layered access control and defense in depth.

2. The CTO wants to add a fourth team next quarter. Walk through what changes in the IAM design and what changes in the bucket policy. Which artifact scales by appending and which forces a rewrite of existing statements? What does that imply for how you would structure the same problem if you knew up front that team count was unbounded? This is a Domain 4 governance-at-scale question.

3. The trust policy `Principal` and the inline policy `Resource` are both common places to make a mistake. What specific error does STS return when the trust policy `Principal` does not match the caller, and what does S3 return when the inline policy `Resource` is wrong but the trust policy is correct? How would you tell those two failure modes apart from the AccessDenied alone? This is a diagnosis question that DEA-C01 likes to surface as a multi-step troubleshooting scenario.

## Exam-Style Practice

**Q1.** A data engineer attaches an inline policy to the analytics role that grants `s3:GetObject` on `arn:aws:s3:::data-lake/*` (every prefix). The bucket policy has an explicit Deny on `s3:GetObject` for the analytics role on the `ml/*` prefix. The role then calls `s3:GetObject` on `s3://data-lake/ml/dataset.csv`. What happens?

A) The call succeeds because identity-based Allows take precedence over resource-based Denies.
B) The call succeeds because `data-lake/*` is a more specific match than `ml/*`.
C) The call is denied because an explicit Deny anywhere in the policy stack overrides every Allow.
D) The call succeeds because AWS evaluates only the inline policy for same-account requests.

<details><summary>Show answer</summary>

**C**.

- A) Wrong because identity-based Allows do not override resource-based Denies; the precedence rule runs the other way.
- B) Wrong because IAM evaluation does not pick the "more specific" match; a single explicit Deny is enough to block the request.
- C) Right because this is the layered defense you built in Task 3. The bucket policy Deny statements close the door even if a future engineer widens the analytics role's inline policy to grant cross-team access.
- D) Wrong because AWS evaluates both identity-based and resource-based policies on every request, including same-account.

</details>

**Q2.** A data lake bucket has prefixes `analytics/`, `ml/`, and `compliance/`. Each team must read and write only its own prefix and never see the others. Which IAM design follows least privilege?

A) One shared role with `s3:GetObject` and `s3:PutObject` on `arn:aws:s3:::bucket/*` and a tag-based ABAC condition that filters by team.
B) One role per team, each with object actions scoped to that team's prefix ARN, plus `s3:ListBucket` scoped via a `Condition` on `s3:prefix`.
C) One admin role plus three IAM users, where each user assumes the admin role with a session policy that filters by prefix.
D) One shared role with full bucket access; revoke access by rotating the role's access keys when isolation is needed.

<details><summary>Show answer</summary>

**B**.

- A) Wrong because tag-based ABAC works against tagged resources, but S3 prefixes are not resources and cannot be tagged individually; the condition has nothing to bind to.
- B) Right because this is the design you built in Task 2. Per-team roles with prefix-scoped Resource ARNs for object actions, plus `s3:ListBucket` scoped via `s3:prefix` Condition, keeps each team in its lane and matches AWS's prescribed prefix-isolation pattern.
- C) Wrong because session policies can only narrow an already-broad role; the underlying admin role is still over-privileged, and a misconfigured caller bypasses the session policy.
- D) Wrong because rotating keys is a manual revocation, not a policy boundary. Least privilege means the role itself cannot reach other teams' data regardless of who holds the credentials.

</details>

**Q3.** A central data team in AWS account A runs a Glue crawler that needs to discover and read an S3 bucket owned by a partner team in account B. What is the correct setup?

A) Embed long-lived IAM user access keys from account B inside the crawler's role configuration in account A.
B) Add a bucket policy on the account B bucket allowing `s3:GetObject` from the crawler role's ARN in account A, with no role created in account B.
C) Create an IAM role in account B whose trust policy lets the crawler's role in account A assume it, grant that account-B role the necessary S3 permissions, and have the crawler call `sts.assume_role` against it.
D) Make the bucket publicly readable so the crawler can fetch objects without any IAM configuration.

<details><summary>Show answer</summary>

**C**.

- A) Wrong because long-lived embedded access keys are an IAM anti-pattern; the assumed-role plus STS flow eliminates persistent cross-account secrets.
- B) Wrong because a bucket-policy-only setup leaves the crawler operating with account A's identity, which complicates cross-account audit trails and does not extend to Glue Data Catalog operations that require resource-level permissions in account B.
- C) Right because this is the canonical cross-account pattern and it generalizes the same trust-plus-inline-plus-AssumeRole flow you built in Task 4. Account B controls who can assume the role via the trust policy; account A's crawler mints fresh temporary credentials for each crawl.
- D) Wrong because a public S3 bucket is a security incident, not a design choice; partner data integrations always go through scoped IAM, never public reads.

</details>
